# Telco Customer Churn — Cleaning Notebook

This notebook loads the raw Telco Customer Churn dataset, performs practical cleaning steps for EDA prep, and saves a cleaned CSV named:

`clean_WA_Fn-UseC_-Telco-Customer-Churn.csv`



In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path("WA_Fn-UseC_-Telco-Customer-Churn.csv")
assert RAW_PATH.exists(), f"Could not find {RAW_PATH.resolve()}"

df = pd.read_csv(RAW_PATH)
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1) Basic column & text cleanup

- Strip whitespace from column names  
- Strip whitespace in string columns  
- Convert blank strings to missing values (`NA`)


In [4]:
# Strip column-name whitespace
df.columns = [c.strip() for c in df.columns]

# Strip and normalize blanks in string columns
obj_cols = df.select_dtypes(include=["object"]).columns
for c in obj_cols:
    df[c] = df[c].astype(str).str.strip()
    df.loc[df[c].isin(["", "nan", "NaN", "None"]), c] = pd.NA

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## 2) Fix datatypes

The raw dataset commonly has `TotalCharges` stored as text (with some blanks). We convert it to numeric.

Also ensure:
- `tenure`, `MonthlyCharges` are numeric  
- `SeniorCitizen` is integer-like (0/1)  


In [5]:
# Convert numeric-like columns
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

for c in ["MonthlyCharges", "tenure"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

if "SeniorCitizen" in df.columns:
    df["SeniorCitizen"] = pd.to_numeric(df["SeniorCitizen"], errors="coerce").astype("Int64")

df[["tenure", "MonthlyCharges", "TotalCharges"]].describe()


,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7032.000000
mean,32.371149,64.761692,2283.300441
std,24.559481,30.090047,2266.771362
min,0.000000,18.250000,18.800000
25%,9.000000,35.500000,401.450000
50%,29.000000,70.350000,1397.475000
75%,55.000000,89.850000,3794.737500
max,72.000000,118.750000,8684.800000


## 3) Remove exact duplicates (if any)

In [6]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
before, after, before - after


(7043, 7043, 0)

## 4) Handle missing values

For this dataset, missing `TotalCharges` is often linked to `tenure == 0`.  
We use a pragmatic strategy useful for EDA:

1. If `tenure == 0` and `TotalCharges` is missing ⇒ set `TotalCharges = 0`  
2. Remaining missing `TotalCharges` ⇒ impute with `MonthlyCharges * tenure` if available 


In [7]:
# TotalCharges special handling
if {"TotalCharges", "tenure"}.issubset(df.columns):
    mask_tc_na = df["TotalCharges"].isna()
    mask_tenure0 = df["tenure"].fillna(-1).eq(0)
    df.loc[mask_tc_na & mask_tenure0, "TotalCharges"] = 0.0

if "TotalCharges" in df.columns:
    still_na = df["TotalCharges"].isna()
    if still_na.any():
        if {"MonthlyCharges", "tenure"}.issubset(df.columns):
            est = df["MonthlyCharges"] * df["tenure"]
            df.loc[still_na, "TotalCharges"] = est.loc[still_na]
        still_na2 = df["TotalCharges"].isna()
        if still_na2.any():
            df.loc[still_na2, "TotalCharges"] = df["TotalCharges"].median()

# Clean customerID
if "customerID" in df.columns:
    df["customerID"] = df["customerID"].astype(str).str.strip()

# Normalize simple Yes/No columns to consistent casing
for c in df.select_dtypes(include=["object"]).columns:
    vals = set(df[c].dropna().unique())
    if vals.issubset({"Yes", "No", "yes", "no"}):
        df[c] = df[c].str.title()

# Sanity check: no negative values
for c in ["MonthlyCharges", "TotalCharges", "tenure"]:
    if c in df.columns:
        df.loc[df[c] < 0, c] = pd.NA


df.isna().sum().sort_values(ascending=False).head(10)


customerID         0
gender             0
SeniorCitizen      0
Partner            0
Dependents         0
tenure             0
PhoneService       0
MultipleLines      0
InternetService    0
OnlineSecurity     0
dtype: int64

## 5) Save cleaned dataset

In [8]:
CLEAN_PATH = RAW_PATH.with_name(f"clean_{RAW_PATH.name}")
df.to_csv(CLEAN_PATH, index=False)

print("Saved:", CLEAN_PATH.resolve())
print("Shape:", df.shape)


Saved: C:\Users\jjoao\OneDrive\Ambiente de Trabalho\Nova\Machine Learning\ML group project\clean_WA_Fn-UseC_-Telco-Customer-Churn.csv
Shape: (7043, 21)
